### ЗАДАЧА: Панель модерации продавцов маркетплейса по паттерну `MVC`

Команда trust & safety маркетплейса проверяет продавцов с рисковыми кейсами.
Для каждого кейса нужно хранить причину проверки, risk score, статус,
ответственного модератора и итоговое решение.

Нужно реализовать внутреннюю консольную панель по паттерну `MVC`, где:
- `Model` хранит кейсы и бизнес-правила,
- `View` отвечает только за вывод,
- `Controller` принимает действия операторов и связывает слои.

НЕОБХОДИМО:

1. Реализовать сущность `ModerationCase`.
2. Реализовать `ModerationModel`, который умеет:
   - добавлять кейс,
   - назначать модератора,
   - переводить кейс в `in_review`,
   - одобрять кейс,
   - отклонять кейс,
   - эскалировать кейс,
   - возвращать список кейсов для отображения.
3. Бизнес-правила модели:
   - кейс нельзя отправить в review без модератора,
   - кейс нельзя approve/reject/escalate, если он не в `in_review`,
   - у финальных кейсов (`approved`, `rejected`, `escalated`) нельзя менять решение.
4. Реализовать `ModerationView` для списка кейсов, успешных сообщений и ошибок.
5. Реализовать `ModerationController`, который вызывает методы модели и передает результат во view.
6. Загрузить начальные кейсы из `initial_cases` в модель.
7. Обработать все действия из `actions` через `controller`.
8. В конце вывести финальную очередь кейсов.


In [4]:
from dataclasses import dataclass

initial_cases = [
    ("MC-100", "best-gadgets", "suspicious_docs", 86),
    ("MC-101", "home-decor-plus", "abnormal_refund_rate", 72),
]

actions = [
    ("show",),
    ("assign", "MC-100", "Olga"),
    ("review", "MC-100"),
    ("escalate", "MC-100"),
    ("create", "MC-102", "sport-zone", "fake_tracking_numbers", 91),
    ("review", "MC-102"),
    ("assign", "MC-102", "Max"),
    ("review", "MC-102"),
    ("reject", "MC-102"),
    ("assign", "MC-101", "Ira"),
    ("review", "MC-101"),
    ("approve", "MC-101"),
    ("approve", "MC-100"),
    ("show",),
]


@dataclass
class ModerationCase:
    case_id: str
    seller: str
    reason: str
    risk_score: int
    status: str = "new"
    moderator: str | None = None
    decision: str | None = None


class ModerationModel:
    def __init__(self):
        # TODO 1: создайте self.cases как пустой словарь
        self.cases = {}

    def add_case(self, case_id: str, seller: str, reason: str, risk_score: int) -> None:
        # TODO 2: если case_id уже существует, вызовите ValueError('Case already exists')
        # TODO 3: создайте ModerationCase и сохраните в self.cases
        if case_id in self.cases:
            raise ValueError('Case already exists')
        self.cases[case_id] = ModerationCase(seller=seller, reason=reason, case_id=case_id, risk_score=risk_score)

    def assign_moderator(self, case_id: str, moderator: str) -> None:
        # TODO 4: если кейс не найден, вызовите ValueError('Case not found')
        # TODO 5: назначьте moderator
        if case_id not in self.cases:
            raise ValueError('Case not found!')
        self.cases[case_id].moderator = moderator

    def send_to_review(self, case_id: str) -> None:
        # TODO 6: если кейс не найден, вызовите ValueError('Case not found')
        # TODO 7: если moderator не назначен, вызовите ValueError('Moderator is required')
        # TODO 8: установите status = 'in_review'
        if case_id not in self.cases:
            raise ValueError('Case not found!')
        if self.cases[case_id].moderator is None:
            raise ValueError('Moderator is required')
        self.cases[case_id].status = 'in_review'
    def approve_case(self, case_id: str) -> None:
        # TODO 9: если кейс не найден, вызовите ValueError('Case not found')
        # TODO 10: если status != 'in_review', вызовите ValueError('Case is not in review')
        # TODO 11: установите status = 'approved' и decision = 'approved'
        if case_id not in self.cases:
            raise ValueError('Case not found')
        if self.cases[case_id].status != 'in_review':
            raise ValueError('Case not in review')
        self.cases[case_id].status = 'approved'
        self.cases[case_id].decision = 'approved'
        

    def reject_case(self, case_id: str) -> None:
        # TODO 12: если кейс не найден, вызовите ValueError('Case not found')
        # TODO 13: если status != 'in_review', вызовите ValueError('Case is not in review')
        # TODO 14: установите status = 'rejected' и decision = 'rejected'
        if case_id not in self.cases:
            raise ValueError('Case not found')
        if self.cases[case_id].status != 'in_review':
            raise ValueError('Case not in review')
        self.cases[case_id].status = 'rejected'
        self.cases[case_id].decision = 'rejected'
        
        

    def escalate_case(self, case_id: str) -> None:
        # TODO 15: если кейс не найден, вызовите ValueError('Case not found')
        # TODO 16: если status != 'in_review', вызовите ValueError('Case is not in review')
        # TODO 17: установите status = 'escalated' и decision = 'escalated'
        if case_id not in self.cases:
            raise ValueError('Case not found')
        if self.cases[case_id].status != 'in_review':
            raise ValueError('Case not in review')
        self.cases[case_id].status = 'escalated'
        self.cases[case_id].decision = 'escalated'

    def list_cases(self) -> list[str]:
        # TODO 18: верните список строк формата:
        #   'case_id | seller | risk_score | status | moderator | decision | reason'
        arr = []
        for el in self.cases.values():
            arr.append(f'{el.case_id} | {el.seller} | {el.risk_score} | {el.status} | {el.moderator} | {el.decision} | {el.reason}')
        return arr

class ModerationView:
    @staticmethod
    def render_cases(rows: list[str]) -> None:
        # TODO 19: если rows пустой, выведите 'Кейсов нет'
        # TODO 20: иначе выведите заголовок 'Очередь модерации:' и затем все строки
        if len(rows) == 0:
            print('Кейсов нет')
        else:
            print('Очередь модерации: ')
            for el in rows:
                print(el)

    @staticmethod
    def render_success(message: str) -> None:
        # TODO 21: выведите сообщение успеха
        print(message)

    @staticmethod
    def render_error(message: str) -> None:
        # TODO 22: выведите сообщение ошибки
        print(message)


class ModerationController:
    def __init__(self, model: ModerationModel, view: ModerationView):
        # TODO 23: сохраните model и view
        self.model = model
        self.view = view

    def create_case(self, case_id: str, seller: str, reason: str, risk_score: int) -> None:
        # TODO 24: через try/except вызовите model.add_case(...)
        # TODO 25: покажите success или error
        try:
            self.model.add_case(case_id, seller, reason, risk_score)
            self.view.render_success('Успешно !')
        except ValueError as e:
            self.view.render_error(e)

    def assign_moderator(self, case_id: str, moderator: str) -> None:
        # TODO 26: через try/except вызовите model.assign_moderator(...)
        # TODO 27: покажите success или error
        try:
            self.model.assign_moderator(case_id, moderator)
            self.view.render_success('Успешно !')
        except ValueError as e:
            self.view.render_error(e)
    def send_to_review(self, case_id: str) -> None:
        # TODO 28: через try/except вызовите model.send_to_review(...)
        # TODO 29: покажите success или error
        try:
            self.model.send_to_review(case_id)
            self.view.render_success('Успешно !')
        except ValueError as e:
            self.view.render_error(e)

    def approve_case(self, case_id: str) -> None:
        # TODO 30: через try/except вызовите model.approve_case(...)
        # TODO 31: покажите success или error
        try:
            self.model.approve_case(case_id)
            self.view.render_success('Успешно !')
        except ValueError as e:
            self.view.render_error(e)

    def reject_case(self, case_id: str) -> None:
        # TODO 32: через try/except вызовите model.reject_case(...)
        # TODO 33: покажите success или error
        try:
            self.model.reject_case(case_id)
            self.view.render_success('Успешно !')
        except ValueError as e:
            self.view.render_error(e)

    def escalate_case(self, case_id: str) -> None:
        # TODO 34: через try/except вызовите model.escalate_case(...)
        # TODO 35: покажите success или error
        try:
            self.model.escalate_case(case_id)
            self.view.render_success('Успешно !')
        except ValueError as e:
            self.view.render_error(e)

    def show_cases(self) -> None:
        # TODO 36: получите данные из model.list_cases() и передайте их во view.render_cases(...)
        self.view.render_cases(self.model.list_cases())


model = ModerationModel()
view = ModerationView()
controller = ModerationController(model, view)

# TODO 37: загрузите начальные кейсы из initial_cases в model
# Подсказка: пройдите по initial_cases циклом for
# и вызовите model.add_case(case_id, seller, reason, risk_score)
for el in initial_cases:
    model.add_case(el[0], el[1], el[2], el[3])

# TODO 38: обработайте все действия из actions
# TODO 39: если action[0] == 'show', вызовите controller.show_cases()
# TODO 40: если action[0] == 'create', распакуйте case_id, seller, reason, risk_score
# и вызовите controller.create_case(...)
# TODO 41: если action[0] == 'assign', распакуйте case_id и moderator
# и вызовите controller.assign_moderator(...)
# TODO 42: если action[0] == 'review', распакуйте case_id
# и вызовите controller.send_to_review(...)
# TODO 43: если action[0] == 'approve', распакуйте case_id
# и вызовите controller.approve_case(...)
# TODO 44: если action[0] == 'reject', распакуйте case_id
# и вызовите controller.reject_case(...)
# TODO 45: если action[0] == 'escalate', распакуйте case_id
# и вызовите controller.escalate_case(...)
for el in actions:
    if el[0] == 'show':
        controller.show_cases()
    elif el[0] == 'create':
        controller.create_case(el[1], el[2], el[3], el[4])
    elif el[0] == 'assign':
        controller.assign_moderator(el[1], el[2])
    elif el[0] == 'review':
        controller.send_to_review(el[1])
    elif el[0] == 'approve':
        controller.approve_case(el[1])
    elif el[0] == 'reject':
        controller.reject_case(el[1])
    elif el[0] == 'escalate':
        controller.escalate_case(el[1])
print()
print("Финальная очередь кейсов:")
# TODO 46: покажите финальную очередь кейсов через controller.show_cases()
controller.show_cases()

Очередь модерации: 
MC-100 | best-gadgets | 86 | new | None | None | suspicious_docs
MC-101 | home-decor-plus | 72 | new | None | None | abnormal_refund_rate
Успешно !
Успешно !
Успешно !
Успешно !
Moderator is required
Успешно !
Успешно !
Успешно !
Успешно !
Успешно !
Успешно !
Case not in review
Очередь модерации: 
MC-100 | best-gadgets | 86 | escalated | Olga | escalated | suspicious_docs
MC-101 | home-decor-plus | 72 | approved | Ira | approved | abnormal_refund_rate
MC-102 | sport-zone | 91 | rejected | Max | rejected | fake_tracking_numbers

Финальная очередь кейсов:
Очередь модерации: 
MC-100 | best-gadgets | 86 | escalated | Olga | escalated | suspicious_docs
MC-101 | home-decor-plus | 72 | approved | Ira | approved | abnormal_refund_rate
MC-102 | sport-zone | 91 | rejected | Max | rejected | fake_tracking_numbers
